#### TODO 
# Future Improvements
# Automatically remove midterm week entries when has_midterm = False 

In [ ]:
# %% GENERATOR 
# ===== IMPORTS =====

from docx import Document
from docx.shared import Pt, Inches 
from docx.enum.text import WD_ALIGN_PARAGRAPH
import os 
from docx.oxml import OxmlElement
from datetime import datetime, timedelta, date
from docx.oxml.ns import qn
from data.configs import *
from shared.loaders import *
from shared.schedule_utils import * 
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT
from docx.oxml import parse_xml
from docx.oxml.ns import nsdecls
from shared.course_schedule import * 

# ====== Helper Functions ======

def get_hyperlink_url(paragraph, hyperlink):

    r_id = hyperlink.get(
        "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
    )

    if not r_id:
        return None

    part = paragraph.part

    if r_id in part.rels:
        return part.rels[r_id].target_ref

    return None

def add_divider(doc):
    p = doc.add_paragraph()
    p_pr = p._p.get_or_add_pPr()

    border = OxmlElement('w:pBdr')
    bottom = OxmlElement('w:bottom')

    bottom.set(qn('w:val'), 'single')   # line style
    bottom.set(qn('w:sz'), '6')         # thickness (6 = subtle)
    bottom.set(qn('w:space'), '1')
    bottom.set(qn('w:color'), 'auto')

    border.append(bottom)
    p_pr.append(border)

    return p

def append_docx_content(target_doc, source_path):

    source_doc = Document(source_path)

    STYLE_MAP = {
        "Body Text": "Normal",
        "List Paragraph": "List Bullet",
        "Heading 1": "Heading 2",
        "Heading 2": "Heading 3"
    }

    for para in source_doc.paragraphs:

        # skip empty paragraphs
        if not para.text.strip():
            continue

        source_style = para.style.name

        # remap styles if needed
        style_name = STYLE_MAP.get(source_style, source_style)

        # fallback if style doesn't exist in template
        if style_name not in target_doc.styles:
            style_name = "Normal"

        new_para = target_doc.add_paragraph(style=style_name)

        # normalize spacing to your template system
        new_para.paragraph_format.space_before = Pt(0)
        new_para.paragraph_format.space_after = Pt(6)

        # copy regular runs / formatting
        for run in para.runs:

            new_run = new_para.add_run(run.text)

            new_run.bold = run.bold
            new_run.italic = run.italic
            new_run.underline = run.underline

        # append hyperlinks separately
        for child in para._p:

            if child.tag.endswith('}hyperlink'):

                url = get_hyperlink_url(para, child)

                if url:

                    link_run = new_para.add_run(f" ({url})")
                    link_run.underline = True

def read_docx_text(filepath):
    doc = Document(filepath)
    return "\n".join(p.text for p in doc.paragraphs)

# ===== HEADER =====

def add_syllabus_header(doc, config):
    doc.add_picture("data/logo.png", width=Inches(2))
    p = doc.add_paragraph(f"{config['course_code']}", style="Title")
    p.paragraph_format.space_after = Pt(0)
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p = doc.add_paragraph(f"{config['course_name']}", style="Title")
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p = doc.add_paragraph(f"{config['term']} | Section {config['section']}")
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    p = add_divider(doc)
    p.paragraph_format.space_before = Pt(6)
    p.paragraph_format.space_after = Pt(6)
    
# ===== Course Information =====

def add_course_information(doc, config):
    doc.add_heading("1. Course Information", level=1)

    p = doc.add_paragraph()
    run = p.add_run("Course: ")
    run.bold = True 
    p.add_run(f"{config['course_code']} {config['section']} {config['course_name']}")
    p.paragraph_format.space_after = Pt(0)

    p = doc.add_paragraph()
    run = p.add_run("Instructor: ")
    run.bold = True
    p.add_run(config['instructor'])
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(0)

    p = doc.add_paragraph()
    run = p.add_run("Email: ")
    run.bold = True
    p.add_run(config['email'])
    p.paragraph_format.space_after = Pt(0)

    p = doc.add_paragraph()
    run = p.add_run("Office Hours: ")
    run.bold = True
    p.add_run(config['office_hours'])

    p = doc.add_paragraph()
    run = p.add_run("Time: ")
    run.bold = True
    p.add_run(config['class_time'])
    p.paragraph_format.space_before = Pt(6)
    p.paragraph_format.space_after = Pt(0)

    p = doc.add_paragraph()
    run = p.add_run("Location: ")
    run.bold = True
    p.add_run(config['location'])

# ===== COURSE OVERVIEW =====

def add_course_overview(doc, config):
    doc.add_heading("2. Course Description", level=1)

    p = doc.add_paragraph()
    p.add_run(f"{config['course_description']}")
    p = doc.add_paragraph()
    p.add_run(f"{config['lecture_hours']}")
    p = doc.add_paragraph()
    run = p.add_run("Prerequisite(s): ")
    run.bold = True
    p.add_run(f"{config['prereqs']}")
    p = doc.add_paragraph()
    p.paragraph_format.space_after = Pt(6)
    run = p.add_run("Antirequisite(s): ")
    run.bold = True
    p.add_run(f"{config['antireqs']}")

    p = doc.add_heading("3. Course Learning Outcomes", level=1)

    for outcome in config["learning_outcomes"]:
        doc.add_paragraph(outcome, style="List Bullet")

def add_textbook(doc, config):
    doc.add_heading("4. Textbook", level=1)

    p = doc.add_paragraph()
    p.add_run(f"{config['textbook']}")
    p = doc.add_paragraph()
    p.add_run(f"{config['textbook_purchase']}")

def add_assessment_table(doc, config):
    doc.add_heading("5. Methods of Evaluation", level=1)

    assignments = config.get("assignments", [])
    schedule = config.get("weeks", [])

    due_info = map_assignment_due_dates(schedule)

    # sort by due week
    assignments = sorted(
        assignments,
        key=lambda a: due_info.get(a["name"], {}).get("week", 999)
    )

    table = doc.add_table(rows=1, cols=5)

    # nicer built-in style
    table.style = "Assessment Table"

    headers = ["Component", "Item", "Weight", "Week", "Due"]

    # ===== HEADER ROW =====

    for i, header in enumerate(headers):

        cell = table.rows[0].cells[i]
        cell.text = header

        # vertical alignment
        cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER

    # ===== DATA ROWS =====

    total_weight = 0
    last_group = None

    for item in assignments:

        row = table.add_row().cells

        group = item.get("type", "Other")
        name = item["name"]
        weight = item["weight"]

        info = due_info.get(name, {})
        due = info.get("date", "—")
        week = str(info.get("week", "—"))

        if isinstance(due, date):
            due = due.strftime("%B %d, %Y")

        if group == "Participation":
            due = "Ongoing"
            week = "—"

        if group == last_group:
            row[0].text = ""
        else:
            row[0].text = group
            last_group = group

        row[1].text = name
        row[2].text = f"{weight}%"
        row[3].text = week
        row[4].text = due

        total_weight += weight

    # ===== TOTAL ROW =====

    row = table.add_row().cells

    row[1].text = "Total"
    row[2].text = f"{total_weight}%"

# ===== POLICIES =====

def add_course_policies(doc, config):

    doc.add_heading("Course Format", level=2)
    for item in config["course_format"]:
        p = doc.add_paragraph(item)
        p.paragraph_format.space_after = Pt(6)

    doc.add_heading("Course Polices", level=2)
    for policy in config["policies"]:
        doc.add_paragraph(policy, style="List Bullet")

# ===== RENDER SCHEDULE =====

def add_schedule_section(doc, config):
    doc.add_heading("6. Tentative Class Schedule", level=1)

    schedule = config["weeks"]

    table = doc.add_table(rows=1, cols=5)
    table.style = "Schedule Table"

    headers = ["Week", "Date", "Topic", "Readings", "Notes"]

    for i, header in enumerate(headers):
        table.rows[0].cells[i].text = header

    for item in schedule:
        row = table.add_row().cells

        week_label = "" if item.get("week") is None else str(item.get("week"))
        date = item["date"].strftime("%b %d")

        row[0].text = week_label
        row[1].text = date
        row[2].text = item.get("topic", "")

        readings = item.get("readings") or ""
        if isinstance(readings, list):
            readings = ", ".join(readings)

        assignments = item.get("assignments") or ""
        if isinstance(assignments, list):
            assignments = ", ".join(assignments)

        row[3].text = readings
        row[4].text = assignments

# ======== ACADEMIC POLICIES =============

def add_policy_appendix(doc, config):
    doc.add_heading("7. FASS Appendix", level=1)
    append_docx_content(doc, POLICY_APPENDIX_FILE)

# ===== MAIN =====

def generate_syllabus(config):
    doc = Document("templates/syllabus_template.docx")

    # 🔑 build document
    add_syllabus_header(doc, config)
    add_course_information(doc, config)
    add_course_overview(doc, config)
    add_textbook(doc, config)
    add_assessment_table(doc, config)
    add_course_policies(doc, config)
    add_schedule_section(doc, config)
    add_policy_appendix(doc, config)

    filename = f"{config['course_code']}_{config['section']}_{config['term']}_Syllabus.docx"
    output_dir = "outputs/syllabi"
    os.makedirs(output_dir, exist_ok=True)

    filepath = os.path.abspath(os.path.join(output_dir, filename))
    doc.save(filepath)

    print(f"Generating syllabus for {config['course_code']} ({config['term']})")
    print(f"Syllabus created: {filename}")
    print("Saving to:", filepath)
    return filepath 

######### Validation #############

def validate_config(config):

    errors = []
    warnings = []
    successes = []

    if config["class_weekday"] not in range(7):
        errors.append("Invalid weekday")

    total_weight = sum(a["weight"] for a in config["assignments"])

    if total_weight != 100:
        errors.append(f"Assignment weights total {total_weight} instead of 100")
    else:
        successes.append("Assignment weights total 100%")

    scheduled_items = len(config["weeks"])

    instructional_weeks = [
        item for item in config["weeks"]
        if item.get("type") == "instruction"
    ]

    midterms = [
        item for item in config["weeks"]
        if item.get("type") == "midterm"
    ]

    yaml_weeks = [
        item for item in instructional_weeks
        if item.get("week") is not None
    ]

    if len(yaml_weeks) != len(instructional_weeks):
        warnings.append("Some instructional items do not have week numbers")

    week_numbers = [item["week"] for item in yaml_weeks]

    if week_numbers != sorted(week_numbers):
        errors.append("Instructional week numbers are not in order")

    if len(week_numbers) != len(set(week_numbers)):
        errors.append("Duplicate instructional week numbers found")

    expected = list(range(1, len(yaml_weeks) + 1))

    if week_numbers != expected: 
        errors.append(
           f"Instructional weeks should be numbered {expected}, found {week_numbers}" 
        )

    if config.get("has_midterm") and len(midterms) == 0:
        errors.append("Course has_midterm=True but no midterm appears in schedule")

    if not config.get("has_midterm") and len(midterms) > 0:
        warnings.append("Schedule includes a midterm but has_midterm=False")

    successes.append(f"Schedule contains {scheduled_items} total scheduled items")
    successes.append(f"Schedule contains {len(instructional_weeks)} instructional weeks")

    return errors, warnings, successes

In [10]:
# %% 🔥 DESIGN / TEST MODE

from shared.course_schedule import build_all_course_schedules
import os

schedules = build_all_course_schedules(
    sections_file="data/f2026_sections.csv",
)

config = schedules[2]

print(f"Testing {config['course_code']} section {config['section']}")

errors, warnings, successes = validate_config(config)

if errors:
    print("ERRORS FOUND:")
    for e in errors:
        print("-", e)
else:
    filepath = generate_syllabus(config)
    print("Generated test syllabus ✔")
    os.startfile(filepath)

Testing MOS 3321 section 550
Generating syllabus for MOS 3321 (F2026)
Syllabus created: MOS 3321_550_F2026_Syllabus.docx
Saving to: c:\Users\klbar\OneDrive - The University of Western Ontario\Faculty\Teaching Infrastructure\outputs\syllabi\MOS 3321_550_F2026_Syllabus.docx
Generated test syllabus ✔


In [ ]:
# %%  🚀 GENERATE ALL FINAL SYLLABI


sections = load_sections_from_csv(
    "data/f2026_sections.csv", 
     COURSE_LOOKUP
    )

generated_files = []

for section in sections:

    config = build_section_config(section)
    print(config["has_midterm"])
    for i, week in enumerate(config["weeks"], start=1):
        print(i, week.get("type", "topic"))

    errors, warnings, successess = validate_config(config)

    if errors:

        print(f"\nSkipping {config['course_code']} section {config['section']}")

        for e in errors:
            print("-", e)

        continue

    filepath = generate_syllabus(config)

    generated_files.append(filepath)

print(f"\nGenerated {len(generated_files)} syllabi ✔")

os.startfile(os.path.abspath("outputs/syllabi"))